# LoRA fine-tune of Stable Diffusion v1.5 — corrected training run

The original notebook's training cell failed with `exit status 2` and no weights
were ever produced. Three separate problems:

1. **Wrong script path.** `train_text_to_image_lora.py` lives in
   `diffusers/examples/text_to_image/`, not `examples/community/`.
2. **Invalid arguments.** `--train_annotation_file` and `--save_every_n_steps`
   do not exist in that script. Passing them is an argparse error, so this alone
   would have failed even with the correct path. The script reads captions from
   a `metadata.jsonl` inside the image folder.
3. **Dead base model.** `runwayml/stable-diffusion-v1-5` was removed from the
   Hub. The community mirror is `stable-diffusion-v1-5/stable-diffusion-v1-5`.

Runtime -> Change runtime type -> **T4 GPU** before running.
Expect roughly 1-2 hours for 2000 steps at 512px on a free T4.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Web-Harvested'
IMAGES = f'{BASE}/valid_images'
CAPTIONS = f'{BASE}/valid_captions.csv'
OUTPUT = f'{BASE}/lora_sd_v1_5'

import os
print('images  :', len(os.listdir(IMAGES)))
print('captions:', os.path.exists(CAPTIONS))


## 1. Build `metadata.jsonl`

The training script loads `--train_data_dir` as a HuggingFace `imagefolder`
dataset, which expects a `metadata.jsonl` beside the images: one JSON object per
line with `file_name` and `text`.

The CSV stores absolute Colab paths, so only the basename is used.


In [ ]:
import csv, json, os

rows, missing = [], 0
with open(CAPTIONS, newline='', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        name = os.path.basename(r['image_path'].strip())
        caption = (r['caption'] or '').strip()
        if not caption:
            continue
        if not os.path.exists(os.path.join(IMAGES, name)):
            missing += 1
            continue
        rows.append({'file_name': name, 'text': caption})

with open(os.path.join(IMAGES, 'metadata.jsonl'), 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + chr(10))

print(f'wrote {len(rows)} pairs, skipped {missing} missing images')
print('sample:', rows[0])


In [ ]:
# Sanity check: the dataset must load before training starts, or you discover
# the problem 40 minutes in.
!pip install -q datasets
from datasets import load_dataset
ds = load_dataset('imagefolder', data_dir=IMAGES, split='train')
print(ds)
print('columns:', ds.column_names)
assert 'text' in ds.column_names, 'metadata.jsonl not picked up'
print('first caption:', ds[0]['text'])


## 2. Install diffusers and the example's requirements

In [ ]:
!git clone -q https://github.com/huggingface/diffusers.git /content/diffusers
%cd /content/diffusers
!pip install -q -e .
%cd /content/diffusers/examples/text_to_image
!pip install -q -r requirements.txt
!pip install -q peft
%cd /content

import os
SCRIPT = '/content/diffusers/examples/text_to_image/train_text_to_image_lora.py'
assert os.path.exists(SCRIPT), 'script not found - check the path'
print('script OK:', SCRIPT)


In [ ]:
from accelerate.utils import write_basic_config
write_basic_config()
print('accelerate configured')


## 3. Train

`--rank 4` keeps the adapter small (~3 MB). Raise to 8 or 16 for more capacity,
at the cost of overfitting risk on 2k images.

Checkpoints are written every 500 steps, so a disconnect does not lose
everything - rerun with `--resume_from_checkpoint latest`.


In [ ]:
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'

!accelerate launch --mixed_precision="fp16" \
  /content/diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="{MODEL_ID}" \
  --train_data_dir="{IMAGES}" \
  --caption_column="text" \
  --resolution=512 --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=2000 \
  --learning_rate=1e-04 \
  --lr_scheduler="cosine" --lr_warmup_steps=0 \
  --max_grad_norm=1 \
  --rank=4 \
  --checkpointing_steps=500 \
  --seed=42 \
  --output_dir="{OUTPUT}" \
  --validation_prompt="a family hugging, stock photo" \
  --validation_epochs=1 \
  --report_to="tensorboard"


In [ ]:
# Verify weights exist. The original run silently produced nothing, so do not
# skip this step.
import os
print(os.listdir(OUTPUT))
w = os.path.join(OUTPUT, 'pytorch_lora_weights.safetensors')
assert os.path.exists(w), 'NO WEIGHTS - training did not complete'
print(f'weights: {os.path.getsize(w)/1e6:.1f} MB')


## 4. Generate a gallery with the fine-tune

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, dtype=torch.float16).to('cuda')
pipe.load_lora_weights(OUTPUT)
pipe.set_progress_bar_config(disable=True)

PROMPTS = [
    'a family of four hugging, warm light, stock photo',
    'ladies summer dress on a pastel background, product photo',
    'seasons of the year represented by an alarm clock, white background',
    'a cosy living room in autumn, natural light',
    'fresh vegetables arranged on a wooden table, top down',
    'a business team meeting in a bright office',
    'pyramids with snow',
    'a lighthouse in a storm, dramatic sky',
]

import os, json
GAL = '/content/drive/MyDrive/Web-Harvested/gallery'
os.makedirs(GAL, exist_ok=True)

manifest = []
for i, p in enumerate(PROMPTS):
    img = pipe(p, num_inference_steps=30, guidance_scale=7.5,
               generator=torch.Generator('cuda').manual_seed(42 + i)).images[0]
    name = f'gen_{i:02d}.png'
    img.save(os.path.join(GAL, name))
    manifest.append({'file': name, 'prompt': p})
    print('saved', name)

with open(os.path.join(GAL, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print('gallery written to', GAL)


## 5. Publish the adapter

Then say so, and the gallery gets wired into the Streamlit demo.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo('MA29/t2i-lora', repo_type='model', exist_ok=True)
api.upload_folder(folder_path=OUTPUT, repo_id='MA29/t2i-lora', repo_type='model')
print('https://huggingface.co/MA29/t2i-lora')
